# StresSense — Dataset Inspection and Preprocessing

This notebook inspects the dataset and creates cross-user training, validation, and test windows. The default `ag` profile uses accelerometer and gyroscope data to match the deployment IMU.

In [1]:
from pathlib import Path
import json
import os

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_DIR)
DATASET = PROJECT_DIR / 'data/raw/StresSense.csv'
print('Working directory:', PROJECT_DIR)
print('Dataset exists:', DATASET.exists(), DATASET)

Working directory: /Users/jaloliddinabdullaev/Projects/activity-recognition-training/stresssense
Dataset exists: True /Users/jaloliddinabdullaev/Projects/activity-recognition-training/stresssense/data/raw/StresSense.csv


In [2]:
import inspect_dataset
inspect_dataset.main()

Dataset: /Users/jaloliddinabdullaev/Projects/activity-recognition-training/stresssense/data/raw/StresSense.csv
Usable rows: 495,446
Users: 37
Activities: 5
Features: ['Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz', 'Mx', 'My', 'Mz']
Resolved columns: {'user': 'User', 'activity': 'Activity', 'Ax': 'Ax', 'Ay': 'Ay', 'Az': 'Az', 'Gx': 'Gx', 'Gy': 'Gy', 'Gz': 'Gz', 'Mx': 'Mx', 'My': 'My', 'Mz': 'Mz'}

Rows per activity:
activity
staying_still    120063
nail_biting      111189
smoking           94586
eating            85188
face_touch        84420

Users per activity:
activity
eating           37
face_touch       30
nail_biting      27
smoking          18
staying_still    35

Users containing every activity:
['1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '2', '3', '4', '5', '6', '7', '8', '9']

Rows per user/activity:
activity  eating  face_touch  nail_biting  smoking  staying_still
user                                                             
1           1772        3245         6983 

## Configuration

- `ag`: accelerometer + gyroscope, compatible with the current wrist IMU
- `agm`: accelerometer + gyroscope + magnetometer
- Window size 250 = 5 seconds at 50 Hz
- One user is reserved for validation and another for final testing

In [4]:
SENSOR_PROFILE = 'ag'
WINDOW_SIZE = 250
OVERLAP = 0.8
TEST_USER = 'auto'
VALIDATION_USER = 'auto'

In [5]:
import sys
import preprocess

sys.argv = [
    'preprocess.py',
    '--sensor-profile', SENSOR_PROFILE,
    '--window-size', str(WINDOW_SIZE),
    '--overlap', str(OVERLAP),
    '--test-user', TEST_USER,
    '--validation-user', VALIDATION_USER,
]
preprocess.main()

STRESSSENSE PREPROCESSING COMPLETE
Sensor profile: ag
Features: ['Ax', 'Ay', 'Az', 'accelerometer_magnitude', 'Gx', 'Gy', 'Gz', 'gyroscope_magnitude']
Validation user: 5
Test user: 4
Training: (8312, 250, 8)
Validation: (426, 250, 8)
Test: (506, 250, 8)
Window: 250 samples
Overlap: 80%


In [6]:
summary = json.loads((PROJECT_DIR / 'data/processed/preprocessing_summary.json').read_text())
summary

{'dataset': 'StresSense',
 'dataset_doi': '10.17632/2dn3hpbm5m.1',
 'sensor_profile': 'ag',
 'sensor_names': ['accelerometer', 'gyroscope'],
 'feature_columns': ['Ax',
  'Ay',
  'Az',
  'accelerometer_magnitude',
  'Gx',
  'Gy',
  'Gz',
  'gyroscope_magnitude'],
 'num_features': 8,
 'sampling_rate': 50,
 'window_size': 250,
 'overlap': 0.8,
 'step_size': 50,
 'test_user': '4',
 'validation_user': '5',
 'activities': ['eating',
  'face_touch',
  'nail_biting',
  'smoking',
  'staying_still'],
 'resolved_columns': {'user': 'User',
  'activity': 'Activity',
  'Ax': 'Ax',
  'Ay': 'Ay',
  'Az': 'Az',
  'Gx': 'Gx',
  'Gy': 'Gy',
  'Gz': 'Gz'},
 'train_shape': [8312, 250, 8],
 'validation_shape': [426, 250, 8],
 'test_shape': [506, 250, 8],
 'train_windows_per_class': {'eating': 1401,
  'face_touch': 1433,
  'nail_biting': 1826,
  'smoking': 1525,
  'staying_still': 2127},
 'test_windows_per_class': {'eating': 88,
  'face_touch': 58,
  'nail_biting': 121,
  'smoking': 176,
  'staying_still': 